<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Yolo_y_Bytetrack_con_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U ultralytics

In [ ]:

from google.colab import files
from ultralytics import YOLO
from collections import defaultdict, Counter
from IPython.display import display, HTML
from pathlib import Path
import cv2
import numpy as np
import os
import time
import subprocess

uploaded = files.upload()

if not uploaded:
    raise ValueError("Debes seleccionar un video.")

video_path = next(iter(uploaded.keys()))

# Modelo YOLO11 entrenado con las clases COCO
model = YOLO("yolo26n.pt")
names = model.names

CONF = 0.25
IMGSZ = 640

# Archivo final
OUTPUT = "/content/video_dashboard_yolo.mp4"

# Seguimiento e historial
ids_por_clase = defaultdict(set)
detecciones = Counter()
eventos = []
historial = []

print("Video seleccionado:", video_path)
print("Clases disponibles:", len(names))
print("El procesamiento se ejecutará sin mostrar fotogramas.")


Saving videoperros.mp4 to videoperros.mp4
Video seleccionado: videoperros.mp4
Clases disponibles: 80
El procesamiento se ejecutará sin mostrar fotogramas.


In [ ]:

# --------------------------------------------
# FUNCIONES PARA EL DASHBOARD
# --------------------------------------------
def texto(img, mensaje, xy, escala=0.55,
          color=(240, 240, 240), grosor=1):
    cv2.putText(
        img, str(mensaje), xy,
        cv2.FONT_HERSHEY_SIMPLEX,
        escala, color, grosor, cv2.LINE_AA
    )

def dibujar_panel(frame, frame_num, total_frames,
                  visibles, ids_visibles, inicio):
    h, w = frame.shape[:2]

    # Panel lateral oscuro
    panel_w = max(300, int(w * 0.34))
    panel = np.zeros((h, panel_w, 3), dtype=np.uint8)
    panel[:] = (28, 28, 28)

    # Cabecera
    cv2.rectangle(panel, (0, 0), (panel_w, 78),
                  (48, 48, 48), -1)
    texto(panel, "YOLO + BYTETRACK", (18, 30),
          0.68, (0, 220, 255), 2)
    texto(panel, "ANALITICA DE OBJETOS", (18, 56),
          0.45)

    # Indicadores principales
    unique_ids = sum(len(v) for v in ids_por_clase.values())
    total_visibles = sum(visibles.values())

    y = 110
    indicadores = [
        ("OBJETOS VISIBLES", total_visibles),
        ("IDS VISIBLES", len(ids_visibles)),
        ("IDS UNICOS OBSERVADOS", unique_ids),
        ("CLASES VISIBLES", len(visibles))
    ]

    for etiqueta, valor in indicadores:
        texto(panel, etiqueta, (18, y), 0.43,
              (175, 185, 195))
        texto(panel, valor, (18, y + 30), 0.85,
              (0, 230, 180), 2)
        y += 66

    # Progreso
    progreso = frame_num / max(total_frames, 1)
    texto(panel, "PROGRESO", (18, y + 5), 0.45)
    bx, by = 18, y + 18
    bw = panel_w - 36
    cv2.rectangle(panel, (bx, by), (bx + bw, by + 14),
                  (65, 65, 65), -1)
    cv2.rectangle(panel, (bx, by),
                  (bx + int(bw * progreso), by + 14),
                  (0, 180, 120), -1)
    texto(panel, f"{progreso * 100:.1f}%",
          (18, by + 38), 0.48)

    # Objetos por clase: ordenados por IDs únicos
    y = by + 75
    texto(panel, "OBJETOS POR CLASE", (18, y),
          0.52, (0, 220, 255), 1)
    y += 24

    clases = sorted(
        ids_por_clase.keys(),
        key=lambda c: len(ids_por_clase[c]),
        reverse=True
    )

    max_items = max(1, (h - y - 100) // 27)
    clases = clases[:max_items]

    max_val = max(
        [len(ids_por_clase[c]) for c in clases] + [1]
    )
    bar_max = int(panel_w * 0.34)

    for clase in clases:
        cantidad = len(ids_por_clase[clase])
        texto(panel, clase[:19], (18, y + 15),
              0.42, (235, 235, 235))

        bar_x = int(panel_w * 0.55)
        bar_y = y + 3
        bar_w = int(bar_max * cantidad / max_val)

        cv2.rectangle(panel, (bar_x, bar_y),
                      (bar_x + bar_max, bar_y + 15),
                      (58, 58, 58), -1)
        if bar_w > 0:
            cv2.rectangle(panel, (bar_x, bar_y),
                          (bar_x + bar_w, bar_y + 15),
                          (0, 180, 120), -1)

        texto(panel, cantidad,
              (panel_w - 27, y + 14), 0.42)
        y += 27

    # Pie
    texto(panel, f"Fotograma {frame_num}/{total_frames}",
          (18, h - 48), 0.43, (190, 190, 190))
    texto(panel, f"Tiempo: {frame_num / FPS:.1f} s",
          (18, h - 25), 0.43, (190, 190, 190))

    return np.hstack((frame, panel))


# --------------------------------------------
# PREPARAR VIDEO
# --------------------------------------------
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir el video.")

FPS = cap.get(cv2.CAP_PROP_FPS)
if not FPS or FPS <= 0:
    FPS = 25.0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
original_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
original_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Reducir videos grandes para agilizar el procesamiento.
MAX_W = 960
scale = min(1.0, MAX_W / original_w)
out_w = int(original_w * scale)
out_h = int(original_h * scale)

panel_w = max(300, int(out_w * 0.34))
final_w = out_w + panel_w

# H.264: MP4 reproducible en la mayoría de los dispositivos.
# Se utiliza FFmpeg mediante un flujo de fotogramas.
cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo",
    "-vcodec", "rawvideo",
    "-pix_fmt", "bgr24",
    "-s", f"{final_w}x{out_h}",
    "-r", str(FPS),
    "-i", "-",
    "-an",
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "23",
    "-pix_fmt", "yuv420p",
    OUTPUT
]

proc = subprocess.Popen(
    cmd,
    stdin=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

frame_num = 0
inicio = time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_num += 1

        if scale < 1:
            frame = cv2.resize(frame, (out_w, out_h))

        # Detectar y actualizar ByteTrack
        result = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=CONF,
            imgsz=IMGSZ,
            verbose=False
        )[0]

        visibles = Counter()
        ids_visibles = set()

        if result.boxes is not None and len(result.boxes):
            boxes = result.boxes.xyxy.cpu().numpy()
            classes = result.boxes.cls.int().cpu().tolist()
            confs = result.boxes.conf.cpu().tolist()

            track_ids = (
                result.boxes.id.int().cpu().tolist()
                if result.boxes.id is not None
                else [None] * len(boxes)
            )

            for box, cls_id, conf, track_id in zip(
                boxes, classes, confs, track_ids
            ):
                x1, y1, x2, y2 = map(int, box)
                clase = names[int(cls_id)]
                visibles[clase] += 1
                detecciones[clase] += 1

                if track_id is not None:
                    track_id = int(track_id)
                    ids_visibles.add(track_id)
                    ids_por_clase[clase].add(track_id)

                    eventos.append({
                        "fotograma": frame_num,
                        "tiempo_segundos": frame_num / FPS,
                        "id": track_id,
                        "clase": clase,
                        "confianza": float(conf),
                        "x1": x1, "y1": y1,
                        "x2": x2, "y2": y2
                    })

                # Color estable por clase
                color = (
                    (37 * int(cls_id) + 70) % 255,
                    (97 * int(cls_id) + 130) % 255,
                    (53 * int(cls_id) + 200) % 255
                )

                cv2.rectangle(
                    frame, (x1, y1), (x2, y2), color, 2
                )

                etiqueta = (
                    f"{clase} ID:{track_id} {conf:.2f}"
                    if track_id is not None
                    else f"{clase} {conf:.2f}"
                )

                texto(
                    frame, etiqueta,
                    (x1, max(20, y1 - 8)),
                    0.48, color, 2
                )

        # Dashboard integrado en cada fotograma
        final_frame = dibujar_panel(
            frame, frame_num, total_frames,
            visibles, ids_visibles, inicio
        )

        proc.stdin.write(
            np.ascontiguousarray(final_frame).tobytes()
        )

        if frame_num % 100 == 0:
            print(
                f"Procesados {frame_num}/{total_frames} "
                f"fotogramas...",
                end="\r"
            )

finally:
    cap.release()
    proc.stdin.close()
    return_code = proc.wait()

if return_code != 0:
    raise RuntimeError(
        "FFmpeg no pudo generar el video. "
        "Comprueba que el entorno tenga FFmpeg instalado."
    )

print(f"\nProcesamiento finalizado en {time.time()-inicio:.1f} s")
print("Video generado:", OUTPUT)



Procesamiento finalizado en 621.2 s
Video generado: /content/video_dashboard_yolo.mp4


In [ ]:

from google.colab import files
import os

if not os.path.exists(OUTPUT) or os.path.getsize(OUTPUT) == 0:
    raise FileNotFoundError("No se generó el video final.")

files.download(OUTPUT)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>